# AWS Glue Studio Notebook
##### You are now running a AWS Glue Studio notebook; To start using your notebook you need to start an AWS Glue Interactive Session.


In [1]:
import sys
import re
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
from awsgluedq.transforms import EvaluateDataQuality
from awsglue.dynamicframe import DynamicFrame

# ✅ Safe JOB_NAME handling
if '--JOB_NAME' in sys.argv:
    args = getResolvedOptions(sys.argv, ['JOB_NAME'])
else:
    args = {'JOB_NAME': 'local_test_job'}

# ✅ Reuse existing SparkContext
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)
job.init(args['JOB_NAME'], args)

DEFAULT_DATA_QUALITY_RULESET = """
    Rules = [
        ColumnCount > 0,
        RowCount > 0
    ]
"""

# ── Regex para limpeza de nomes de colunas ─────────────────────────

regex_pattern = r"[.,:\s()~^´`{}\[\]!/'\"\-@#$%&*+=<>?|\\]"

def clean_column_name(col_name):
    cleaned = re.sub(regex_pattern, '_', col_name)
    cleaned = re.sub(r'_+', '_', cleaned)
    cleaned = cleaned.strip('_')
    return cleaned

def rename_columns(dynamic_frame, glueContext, ctx_name):
    df = dynamic_frame.toDF()
    original_cols = df.columns
    new_cols = [clean_column_name(c) for c in original_cols]

    for orig, new in zip(original_cols, new_cols):
        if orig != new:
            (f"  [{ctx_name}] Renomeando: '{orig}' → '{new}'")

    for orig, new in zip(original_cols, new_cols):
        df = df.withColumnRenamed(orig, new)

    return DynamicFrame.fromDF(df, glueContext, ctx_name)


# ── Helper principal ───────────────────────────────────────────────

def process_dataset(glueContext, frame, dq_context, output_path, transformation_ctx):
    # Cache e materialização do frame
    df = frame.toDF().cache()
    count = df.count()
    print(f"\n[{dq_context}] Row count: {count}")

    if count == 0:
        print(f"[{dq_context}] ⚠️ Frame vazio - pulando DQ e escrita.")
        return

    # Limpa nomes de colunas
    dynamic_frame = DynamicFrame.fromDF(df, glueContext, transformation_ctx)
    dynamic_frame = rename_columns(dynamic_frame, glueContext, dq_context)

    print(f"[{dq_context}] Schema após limpeza:")
    dynamic_frame.printSchema()

    # ✅ Executa DQ com apply() — compatível com notebooks e jobs
    try:
        dq_results = EvaluateDataQuality.apply(
            frame=dynamic_frame,
            ruleset=DEFAULT_DATA_QUALITY_RULESET,
            publishing_options={
                "dataQualityEvaluationContext": dq_context,
                "enableDataQualityResultsPublishing": True,
                "enableDataQualityCloudWatchMetrics": True
            }
        )

        # Exibe resultado das regras
        rule_outcomes = dq_results.select_fields(["Rule", "Outcome", "FailureReason"])
        (f"[{dq_context}] Resultado do Data Quality:")
        rule_outcomes.toDF().show(truncate=False)

        # Verifica se alguma regra falhou
        failed = rule_outcomes.toDF().filter("Outcome = 'Failed'").count()
        if failed > 0:
            (f"[{dq_context}] ⚠️ {failed} regra(s) falharam — verifique os dados.")
        else:
            (f"[{dq_context}] ✅ Todas as regras de DQ passaram.")

    except Exception as e:
        (f"[{dq_context}] ⚠️ Data Quality falhou (não bloqueante): {e}")

    # ✅ Escreve em S3 como Parquet
    glueContext.write_dynamic_frame.from_options(
        frame=dynamic_frame,
        connection_type="s3",
        format="glueparquet",
        connection_options={"path": output_path, "partitionKeys": []},
        format_options={"compression": "snappy"},
        transformation_ctx=transformation_ctx
    )
    (f"[{dq_context}] ✅ Escrito em: {output_path}")


# ── Leitura dos arquivos fonte ─────────────────────────────────────

Arquivo_Pesquisa2526 = glueContext.create_dynamic_frame.from_options(
    format_options={"quoteChar": "\"", "withHeader": True, "separator": ",", "optimizePerformance": False},
    connection_type="s3", format="csv",
    connection_options={"paths": ["s3://lab654487690909/data_input/pesquisa_2526/Final_Dataset_State_Data_2025_2026.csv"], "recurse": True},
    transformation_ctx="Arquivo_Pesquisa2526"
)

Arquivo_Pesquisa2324 = glueContext.create_dynamic_frame.from_options(
    format_options={"quoteChar": "\"", "withHeader": True, "separator": ",", "optimizePerformance": False},
    connection_type="s3", format="csv",
    connection_options={"paths": ["s3://lab654487690909/data_input/pesquisa_2324/State_of_data_BR_2023_Kaggle.csv"], "recurse": True},
    transformation_ctx="Arquivo_Pesquisa2324"
)

Arquivo_Pesquisa22 = glueContext.create_dynamic_frame.from_options(
    format_options={"quoteChar": "\"", "withHeader": True, "separator": ",", "optimizePerformance": False},
    connection_type="s3", format="csv",
    connection_options={"paths": ["s3://lab654487690909/data_input/pesquisa_22/State_of_data_2022.csv"], "recurse": True},
    transformation_ctx="Arquivo_Pesquisa22"
)

# ── Processamento de cada dataset ─────────────────────────────────

process_dataset(
    glueContext,
    Arquivo_Pesquisa2526,
    "EvaluateDataQuality_pesq2526",
    "s3://lab654487690909/data_output/pesquisa_2526/",
    "tb_pesq2526"
)

process_dataset(
    glueContext,
    Arquivo_Pesquisa2324,
    "EvaluateDataQuality_pesq2324",
    "s3://lab654487690909/data_output/pesquisa_2324/",
    "tb_pesq2324"
)

process_dataset(
    glueContext,
    Arquivo_Pesquisa22,
    "EvaluateDataQuality_pesq22",
    "s3://lab654487690909/data_output/pesquisa_22/",
    "tb_pesq22"
)

job.commit()


Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Trying to create a Glue session for the kernel.
Session Type: glueetl
Session ID: 74700049-99ba-4d80-8f42-8cb4a570aec0
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
Waiting for session 74700049-99ba-4d80-8f42-8cb4a570aec0 to get into ready status...
Session 74700049-99ba-4d80-8f42-8cb4a570aec0 has been created.

[EvaluateDataQuality_pesq2526] Row count: 3495
[EvaluateDataQuality_pesq2526] Schema após limpeza:
root
|-- 0_a_token: string
|-- 0_d_data_hora_envio: string
|-- 1_a_idade: string
|-- 1_a_1_faixa_idade: string
|-- 1_b_genero: string
|-- 1_c_cor_raca_etnia: string
|-- 1_d_pcd: s

In [4]:
import sys
import re
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
from awsglue.dynamicframe import DynamicFrame
from pyspark.sql import DataFrame

# ✅ Safe JOB_NAME handling
if '--JOB_NAME' in sys.argv:
    args = getResolvedOptions(sys.argv, ['JOB_NAME'])
else:
    args = {'JOB_NAME': 'local_test_job'}

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)
job.init(args['JOB_NAME'], args)


# ── Função de sanitização de nomes de colunas ─────────────────────

def sanitize_column_names(df: DataFrame, ctx_name: str) -> DataFrame:
    """
    Substitui caracteres especiais nos nomes das colunas por '_'.
    Resolve AnalysisException causado por '.', '/', '-', espaços, etc.
    Garante nomes únicos após a sanitização.
    """
    def clean_name(name: str) -> str:
        return re.sub(r'[^a-zA-Z0-9_]', '_', name)

    new_columns = [clean_name(c) for c in df.columns]

    # ✅ Garante unicidade em caso de colisão após limpeza
    seen = {}
    unique_columns = []
    for name in new_columns:
        if name in seen:
            seen[name] += 1
            unique_columns.append(f"{name}_{seen[name]}")
        else:
            seen[name] = 0
            unique_columns.append(name)

    # ✅ Apenas totais — sem detalhe de nomes
    total_renamed = sum(1 for orig, new in zip(df.columns, unique_columns) if orig != new)
    if total_renamed > 0:
        print(f"[{ctx_name}] 🔧 Colunas corrigidas (caracteres especiais): {total_renamed}")
    else:
        print(f"[{ctx_name}] ✅ Nenhuma coluna precisou ser corrigida.")

    return df.toDF(*unique_columns)


# ── Leitura dos arquivos Parquet do data_output ────────────────────

Parquet_Pesquisa2526 = glueContext.create_dynamic_frame.from_options(
    connection_type="s3",
    format="parquet",
    connection_options={"paths": ["s3://lab654487690909/data_output/pesquisa_2526/"], "recurse": True},
    transformation_ctx="Parquet_Pesquisa2526"
)

Parquet_Pesquisa2324 = glueContext.create_dynamic_frame.from_options(
    connection_type="s3",
    format="parquet",
    connection_options={"paths": ["s3://lab654487690909/data_output/pesquisa_2324/"], "recurse": True},
    transformation_ctx="Parquet_Pesquisa2324"
)

Parquet_Pesquisa22 = glueContext.create_dynamic_frame.from_options(
    connection_type="s3",
    format="parquet",
    connection_options={"paths": ["s3://lab654487690909/data_output/pesquisa_22/"], "recurse": True},
    transformation_ctx="Parquet_Pesquisa22"
)


# ── Função de limpeza: remove duplicatas e colunas nulas ───────────

def clean_dataset(glueContext, frame, ctx_name, output_path, transformation_ctx):

    # ── 0. Converte para DataFrame e sanitiza nomes de colunas ────
    df = sanitize_column_names(frame.toDF(), ctx_name)
    df = df.cache()

    total_rows = df.count()
    total_cols = len(df.columns)
    print(f"[{ctx_name}] Rows originais  : {total_rows}")
    print(f"[{ctx_name}] Colunas totais  : {total_cols}")

    # ── 1. Remove colunas onde TODOS os valores são nulos ──────────
    non_null_cols = [
        c for c in df.columns
        if df.filter(df[c].isNotNull()).count() > 0
    ]
    total_removed_cols = total_cols - len(non_null_cols)

    if total_removed_cols > 0:
        print(f"[{ctx_name}] 🗑️ Colunas removidas (100% nulas): {total_removed_cols}")
    else:
        print(f"[{ctx_name}] ✅ Nenhuma coluna 100% nula encontrada.")

    df = df.select(non_null_cols)

    # ── 2. Remove linhas duplicadas ────────────────────────────────
    df_dedup = df.dropDuplicates()
    removed_rows = total_rows - df_dedup.count()

    if removed_rows > 0:
        print(f"[{ctx_name}] 🗑️ Linhas duplicadas removidas: {removed_rows}")
    else:
        print(f"[{ctx_name}] ✅ Nenhuma linha duplicada encontrada.")

    print(f"[{ctx_name}] Rows após limpeza   : {df_dedup.count()}")
    print(f"[{ctx_name}] Colunas após limpeza: {len(df_dedup.columns)}")

    # ── 3. Converte e escreve em S3 ────────────────────────────────
    dynamic_frame_clean = DynamicFrame.fromDF(df_dedup, glueContext, transformation_ctx)

    glueContext.write_dynamic_frame.from_options(
        frame=dynamic_frame_clean,
        connection_type="s3",
        format="glueparquet",
        connection_options={"path": output_path, "partitionKeys": []},
        format_options={"compression": "snappy"},
        transformation_ctx=transformation_ctx
    )
    print(f"[{ctx_name}] ✅ Escrito em: {output_path}")

    df.unpersist()


# ── Processamento de limpeza de cada dataset ───────────────────────

clean_dataset(
    glueContext,
    Parquet_Pesquisa2526,
    "Clean_pesq2526",
    "s3://lab654487690909/data_output_clean/pesquisa_2526/",
    "tb_pesq2526_clean"
)

clean_dataset(
    glueContext,
    Parquet_Pesquisa2324,
    "Clean_pesq2324",
    "s3://lab654487690909/data_output_clean/pesquisa_2324/",
    "tb_pesq2324_clean"
)

clean_dataset(
    glueContext,
    Parquet_Pesquisa22,
    "Clean_pesq22",
    "s3://lab654487690909/data_output_clean/pesquisa_22/",
    "tb_pesq22_clean"
)

job.commit()


[Clean_pesq2526] 🔧 Colunas corrigidas (caracteres especiais): 1
[Clean_pesq2526] Rows originais  : 13980
[Clean_pesq2526] Colunas totais  : 4
[Clean_pesq2526] ✅ Nenhuma coluna 100% nula encontrada.
[Clean_pesq2526] 🗑️ Linhas duplicadas removidas: 8501
[Clean_pesq2526] Rows após limpeza   : 5479
[Clean_pesq2526] Colunas após limpeza: 4
[Clean_pesq2526] ✅ Escrito em: s3://lab654487690909/data_output_clean/pesquisa_2526/
[Clean_pesq2324] 🔧 Colunas corrigidas (caracteres especiais): 399
[Clean_pesq2324] Rows originais  : 21172
[Clean_pesq2324] Colunas totais  : 399
[Clean_pesq2324] ✅ Nenhuma coluna 100% nula encontrada.
[Clean_pesq2324] 🗑️ Linhas duplicadas removidas: 15878
[Clean_pesq2324] Rows após limpeza   : 5294
[Clean_pesq2324] Colunas após limpeza: 399
[Clean_pesq2324] ✅ Escrito em: s3://lab654487690909/data_output_clean/pesquisa_2324/
[Clean_pesq22] 🔧 Colunas corrigidas (caracteres especiais): 353
[Clean_pesq22] Rows originais  : 17084
[Clean_pesq22] Colunas totais  : 353
[Clean_pe